# Suite No. 2 Upgrade: Evoked L4 Drive Readout & Baseline Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_suite_no_2_evoked_l4_drive.ipynb)

## Learning objectives

1. Configure and simulate baseline vs evoked L4 drive conditions in a V1 column.
2. Generate stimulus-locked readouts (raster, LFP-proxy, CSD-proxy, spectral).
3. Compare baseline and evoked response profiles using package-native paradigm API.
4. Interpret response latency, amplitude, and spectral changes as proxy diagnostics.
5. Validate finite outputs and JSON-safe reports in computational scaffold context.

## Question

How does a localized L4 feedforward drive modulate laminar firing and field readouts in a reduced V1 column?


## Setup

```python
import jaxfne as jtfne
import numpy as np

# Baseline configuration
cfg_baseline = (jtfne.Configuration()
    .runtime(seed=7, dtype="float32", duration_ms=1500.0, dt_ms=0.1)
    .column("V1_reduced", layers=["L2/3", "L4", "L5"], n=100)
    .cell_type_drives({"E": 8.0, "PV": 4.0})
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "LFP-proxy", "CSD-proxy"]))

# Evoked configuration: add L4 drive paradigm
paradigm = jtfne.evoked_l4_drive_paradigm(
    l4_onset_ms=500.0,
    l4_duration_ms=200.0,
    l4_amplitude=1.0,
    pre_stimulus_buffer_ms=200.0,
    post_stimulus_buffer_ms=500.0,
)

cfg_evoked = cfg_baseline.paradigm(paradigm)

# Simulate
model_baseline = jtfne.construct(cfg_baseline)
model_evoked = jtfne.construct(cfg_evoked)

signals_baseline = jtfne.simulate(model_baseline, seed=7, duration_ms=1500.0, dt_ms=0.1)
signals_evoked = jtfne.simulate(model_evoked, seed=7, duration_ms=1500.0, dt_ms=0.1)
```

## Mathematical glossary

### Stimulus schedule & event timing

**Formal equation**:
$$I_{L4}(t) = \begin{cases}
A & \text{if } t_{\text{onset}} \le t < t_{\text{onset}} + \tau_{\text{dur}} \\
0 & \text{otherwise}
\end{cases}$$

**Terms**:
- $I_{L4}(t)$: L4 feedforward drive current (relative units)
- $A$: amplitude (1.0 baseline)
- $t_{\text{onset}}$: stimulus onset (500 ms in trial)
- $\tau_{\text{dur}}$: stimulus duration (200 ms)

**Worded equation**: L4 drive is a rectangular pulse at a fixed onset and duration within a trial.

**Implementation location**: `jaxfne.evoked_l4_drive_paradigm()`

**Scope boundary**: Declarative timing only. No real sensory encoding.

---

### Stimulus-locked averaging & response window

**Formal equation**:
$$\bar{R}_{\text{evoked}}[t] = \frac{1}{N_{\text{trials}}} \sum_{n=1}^{N_{\text{trials}}} R_n[t + t_{\text{onset}}]$$

**Terms**:
- $\bar{R}_{\text{evoked}}[t]$: trial-averaged response (raster or LFP) aligned to stimulus onset
- $R_n[t]$: response trace on trial $n$
- $t_{\text{onset}}$: stimulus onset time
- $N_{\text{trials}}$: number of trials (or conditions)

**Worded equation**: Evoked response is computed by aligning and averaging individual trial recordings to the stimulus onset time.

**Implementation location**: `jaxfne.vis.raster()`, `jtfne.vis.lfp_traces()`

**Scope boundary**: Alignment is computational; no physiological latency calibration.


## Figures (placeholders)

- **Figure 1: Stimulus schedule** — Timeline showing baseline and evoked L4 drive onset/duration.
- **Figure 2: Baseline raster & rate** — Spiking activity before stimulus onset.
- **Figure 3: Evoked raster & rate** — Spiking activity during and after L4 drive.
- **Figure 4: LFP-proxy response** — Laminar potential profile baseline vs evoked.
- **Figure 5: CSD-proxy response** — Current source density contrast, baseline vs evoked.
- **Figure 6: Spectral summary** — Power in alpha/beta/gamma bands, condition comparison.

These figures are generated by running the notebook.

## Failure modes & exercises

**Failure modes**:
1. **No visible response**: Drive amplitude too low, or L4 cells silent due to tonic suppression.
2. **Excessive synchrony**: E-to-E coupling too strong; circuit locks into synchronous bursting.
3. **Inverted phase**: CSD polarity reversed due to sink/source geometry; check layer assignment.
4. **Numerical instability**: High drive + recurrent feedback → NaN/inf; reduce drive or add regularization.

**Exercises**:
1. Vary L4 drive amplitude (0.5, 1.0, 2.0) and plot response gain curve.
2. Change L4 stimulus onset (100, 300, 500 ms) and measure latency shift.
3. Add feedback from L5 to L4 and compare evoked vs baseline phase shift.
4. Run with smaller column (n=50) to test scale invariance.
5. Compare float32 vs float64 (enable_x64) output stability.

## Coverage boundary

This tutorial covers:
- Declarative paradigm specification (baseline vs evoked conditions)
- Reduced Izhikevich column with layer anatomy
- Package-native simulation and readout
- Stimulus-locked proxy readouts (raster, LFP, CSD)
- Baseline-evoked comparison as computational diagnostic

This tutorial does **NOT** cover:
- Real sensory encoding or psychophysics
- Empirical parameter fitting
- Biophysical validation against data
- Mechanism proof (responses are proxy readouts only)
- Amplitude calibration
- Non-reduced emitter models (use Jaxley for detailed HH)
